# LLM Request Quickstart

LLM 호출 + MongoDB 캐시를 한 번에 처리하는 `llm_request` 사용 가이드입니다.

| 함수 | 설명 |
| --- | --- |
| `llm_request` | LLM 호출 + 캐시 (sync) |
| `llm_request_async` | LLM 호출 + 캐시 (async) |
| `llm_request_stream` | 스트리밍 LLM 호출 + 캐시 (sync) |
| `llm_request_stream_async` | 스트리밍 LLM 호출 + 캐시 (async) |
| `done_check` | 입력을 해시하여 캐시 키 생성 + DB 조회 |

**사전 조건**
- LLM Handler 서비스 실행 중 (`http://192.7.9.45:8100`)
- MongoDB 접근 가능 (`mongodb://192.7.9.45:27017/`)

## 설정

In [13]:
!pip install -e ../llm-handler/sdk
!pip install -e ../mdb-handler/sdk
!pip install -e .

Obtaining file:///home/rladustn1227/02.github/Kiwoom-Rag/packages/llm-handler/sdk
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for kiwoom-llm-handler (pyproject.toml) ... done
  Created wheel for kiwoom-llm-handler: filename=kiwoom_llm_handler-0.1.0-0.editable-py3-none-any.whl size=3011 sha256=a6d83041de4869e20dca011fdef0b9b380eb88459e7360340aacde1a4aa6c142
  Stored in directory: /tmp/pip-ephem-wheel-cache-rt3aya_l/wheels/88/98/24/06e166ae53c85321a750fa6153af7d4b0573235376cb2b3cd9
Successfully built kiwoom-llm-handler
  Attempting uninstall: kiwoom-llm-handler
    Found existing installation: kiwoom-llm-handler 0.1.0
    Uninstalling kiwoom-llm-handler-0.1.0:
      Successfully uninstalled kiwoom-llm-handler-0.1.0
Obtaining file:///home/rladustn1227/02.github/Kiwoom-Rag/packages/mdb-handler/sdk
  Installin

In [14]:
# import logging
# logging.basicConfig(
#     level=logging.INFO,
#     format="%(name)s - %(levelname)s - %(message)s",
# )

In [1]:
import os

os.environ["MONGODB_URI"] = "mongodb://192.7.9.45:27017/"
os.environ["MONGODB_DATABASE"] = "kw_ai_dev"
# os.environ["LLM_HANDLER_URL"] = "http://host.docker.internal:8100"
os.environ["LLM_HANDLER_URL"] = "http://192.7.9.45:8100"

from mdb_handler import MongoDBHandler

MongoDBHandler.reset()

import llm_handler

llm_handler.configure()

import llm_handler.aio

llm_handler.aio.configure()

print("설정 완료")

설정 완료


> **⚠️ 데이터베이스 안내**
>
> 이 Quickstart에서는 **`kw_ai_dev`** 데이터베이스에 저장됩니다 (개발/테스트용).
> 실제 에이전트 구축 시에는 **`kw_ai`** 로 변경하여 저장해야 합니다.
>
> ```python
> # 개발/테스트
> os.environ["MONGODB_DATABASE"] = "kw_ai_dev"
>
> # 운영/에이전트
> os.environ["MONGODB_DATABASE"] = "kw_ai"
> ```

---
## 1. `llm_request` — 단건 LLM 호출 + 캐시 (sync)

`llm_request`는 첫 인자 `llm_input`(dict)과 선택 키워드 인자(`cache_mode`, `save_mode`, …)로 LLM 호출, 캐시 조회, DB 저장을 처리합니다.

### 호출 형태

```python
response, meta = llm_request(
    {                       # llm_input: llm_handler.call()에 전달할 인자
        "contents": "질문", # 필수. str 또는 [str | Part | FileRef] 혼합 리스트
        "model": "gemini-3-flash-preview",
        "system": "시스템 프롬프트",   # 선택
        "temperature": 0,             # 선택
        "max_tokens": 200,            # 선택
        "service_name": "my_service", # 선택 (메타에 기록)
    },
    cache_mode=True,         # 캐시 조회 여부 (기본 True)
    save_mode=True,          # DB 저장 여부 (기본 True)
    ttl_seconds=None,        # 캐시 만료 (선택)
    debug=False,             # route_reason (선택)
)
```


### 1-1. 기본 호출 (캐시 + 저장 모두 켬)

In [2]:
from llm_request import llm_request

llm_input = {
    "contents": "대한민국의 수도는?",
    "model": "gemini-3-flash-preview",
    "temperature": 0,
    "max_tokens": 200,
}

response, meta = llm_request(llm_input)  # sync — await 불필요
print(response.text)
print(f"source: {meta['source']}")  # 'live' 또는 'cache'


대한민국의 수도는 **서울**입니다.
source: cache


### 1-2. 동일 입력 재호출 → 캐시 히트

In [3]:
response2, meta2 = llm_request(llm_input)

print(f"source: {meta2['source']}")       # 'cache'
print(f"cache_hit: {meta2['cache_hit']}")  # True
print(f"text: {response2.text}")


source: cache
cache_hit: True
text: 대한민국의 수도는 **서울**입니다.


### 1-3. 캐시 없이 LLM만 호출

In [4]:
llm_input_no_cache = {
    "contents": "1+1+1은?",
    "model": "gemini-3-flash-preview",
    "temperature": 0,
    "max_tokens": 200,
}

response3, meta3 = llm_request(
    llm_input_no_cache,
    cache_mode=False,
    save_mode=True,
)
print(response3.text)
print(f"cache_key: {meta3['cache_key']}")


1+1+1은 **3**입니다.
cache_key: 6f67e343efb175004004e085692d6b7f232311779e1ad4eeeba84a57be05fc28


### 1-4. 시스템 프롬프트 + service_name 지정

In [5]:
llm_input_system = {
    "contents": "ESG 평가 기준을 알려줘",
    "model": "gemini-3-flash-preview",
    "system": "키움증권 ESG 전문가입니다. 간결하게 답변하세요.",
    "service_name": "esg_assistant",
    "max_tokens": 500,
}

response4, meta4 = llm_request(llm_input_system)
print(response4.text)
print(f"metadata: {meta4['metadata']}")  # service_name, model 기록됨


키움증권 ESG 전문가입니다. 주요 ESG 평가 기관(KCGS, MSCI 등)의 공통적인 평가 기준은 다음과 같습니다.

**1. 환경(E): 기후변화 대응 능력**
*   **탄소 배출:** 온실가스 배출량 및 감축 목표
*   **에너지 효율:** 재생에너지 사용 비중 및 자원 순환
*   **친환경 제품:** 환경 보호를 위한 제품 및 서비스 개발

**2. 사회(S): 이해관계자와의 상생**
*   **인적자본:** 고용 평등, 교육 훈련, 안전 보건
*   **공급망 관리:** 협력사 ESG 리스크 관리
*   **인권 및 지역사회:** 인권 준수 및 사회공헌 활동

**3. 지배구조(G): 투명한 경영 체계**
*   **이사회 구조:** 사외이사 독립성, 다양성, 전문성
*   **주주 권리:** 주주 환원 정책, 공시 투명성
*   **윤리 경영:** 뇌물 방지, 내부 통제 및 감사 시스템

각 기업이 속한 산업군에 따라 항목별 가중치는 차등 적용
metadata: {'service_name': 'esg_assistant', 'model': 'gemini-3-flash-preview'}


### 1-5. 반환값 구조

모든 `llm_request` 계열 함수는 `(LLMResponse, meta_dict)` 튜플을 반환합니다.

```python
response, meta = llm_request(llm_input)

# response: LLMResponse
print(response.text)          # 모델 응답 텍스트
print(response.model)         # 사용된 모델명
print(response.finish_reason) # "STOP" | "MAX_TOKENS" | "SAFETY" 등
if response.usage:
    u = response.usage
    print(u.input_tokens, u.output_tokens, u.total_tokens)
    print(u.thinking_tokens)  # thinking 모드 사용 시에만 존재

# meta: dict
{
    "source":     "live" | "cache",
    "cache_hit":  bool,
    "cache_key":  str | None,   # SHA-256 해시 (64자)
    "meta":       dict,         # {service_name, model, provider}
    "collection": str | None,   # 저장된 MongoDB 컬렉션명
    # debug=True 시 추가
    "route_reason": str,        # "cache_hit" | "cache_miss" | "cache_off_save_off" 등
}
```


---
## 2. `llm_request_async` — 단건 LLM 호출 + 캐시 (async)

`llm_request`와 동일한 기능의 async 버전입니다. `await`로 호출합니다.
복합 파이프라인(LLM 여러 번 순차 호출)에서 `await`로 이벤트 루프에 양보하여 다른 작업과 병렬 실행할 때 사용합니다.

In [6]:
from llm_request import llm_request_async

response_async, meta_async = await llm_request_async(
    {
        "contents": "파이썬의 장점을 한 줄로 알려줘",
        "model": "gemini-3-flash-preview",
        "temperature": 0,
        "max_tokens": 300,
    },
)
print(response_async.text)
print(f"source: {meta_async['source']}")


문법이 직관적이라 배우기 쉽고, 방대한 라이브러리 덕분에 개발 생산성이 매우 높습니다.
source: cache


---
## 3. 파일 포함 — file_paths

`file_paths`에 파일 경로를 넘기면 크기에 따라 자동으로 처리된다.

| 조건 | 처리 방식 |
|------|----------|
| ≤20MB | inline (base64) |
| >20MB | Files API 업로드 |
| PDF >50MB | split → Files API 업로드 |

> 내부 동작 상세는 README의 **8. File Handler** 참고


In [7]:
from pathlib import Path
from llm_request import llm_request_async

# quickstart.pdf 경로 탐색
for _p in (Path("quickstart.pdf"), Path("../quickstart.pdf")):
    if _p.exists():
        SAMPLE_PDF_SMALL = str(_p.resolve())
        break
else:
    raise FileNotFoundError("quickstart.pdf를 찾을 수 없습니다")

# 대용량 PDF 경로 (아모레퍼시픽)
SAMPLE_PDF = str(Path("../../../Kiwoom-ESG/pdf_files/2024_아모레퍼시픽.pdf").resolve())


In [8]:
# 3-1. inline (≤20MB)
response, meta = await llm_request_async(
    {"contents": "이 PDF를 한글로 요약해줘", "model": "gemini-2.5-flash", "max_tokens": 1000},
    file_paths=[SAMPLE_PDF_SMALL],
    database="kw_ai_dev",
)
print(response.text[:200])
print(f"source: {meta['source']}")

다음은 제공된 PDF 문서 "2024 KB금융그룹 지속가능경영보고서 - 투자자 관점"에 대한 요약입니다.

---

### 2
source: cache


In [9]:
# 3-2. Files API (>50MB PDF → 자동 split + 업로드)
response2, meta2 = await llm_request_async(
    {"contents": "이 PDF의 제목을 한글의 한 줄로.", "model": "gemini-2.5-flash", "max_tokens": 1000},
    file_paths=[SAMPLE_PDF],
    database="kw_ai_dev",
)
print(response2.text)
print(f"source: {meta2['source']}")


2024 아모레퍼시픽홀딩스 지속가능성 보고서
source: cache


In [10]:
# 3-3. Context Cache (같은 파일로 반복 호출 시 토큰 비용 절감)
response3, meta3 = await llm_request_async(
    {"contents": "ESG 등급은?", "model": "gemini-2.5-flash", "max_tokens": 1000},
    file_paths=[SAMPLE_PDF],
    context_cache_ttl=3600,
    database="kw_ai_dev",
)
print(response3.text[:200])
print(f"source: {meta3['source']}")


2024년 기준으로 아모레퍼시픽홀딩스와 아모레퍼시픽의 ESG 등급은 다음과 같습니다 (페이지 6 참고):

**아모레퍼시픽홀딩스 (AMOREPACIFIC Holdings)**
*   **한국ESG기준원:** A+
*   **MSCI:** AA
*   **FTSE4Good:** 13년 연속 편입

**아모레퍼시픽 (AMOREPACIFIC)**
*   **한국
source: cache


In [11]:
# 3-4. 테스트 데이터 정리
from llm_request.file_handler import cache_store
from llm_request.checksum import file_checksum

for pdf_path in [SAMPLE_PDF_SMALL, SAMPLE_PDF]:
    cs = file_checksum(pdf_path)["checksum"]
    cache_store.delete_files_api(cs)
    print(f"삭제: {cs[:20]}...")
print("정리 완료")


삭제: sha256:e6fdb7ac05ee8...
삭제: sha256:0a4061daf4881...
정리 완료


---
## 4. `llm_request_stream` — 스트리밍 LLM 호출 (sync)

LLM 응답을 chunk 단위로 실시간 수신합니다. 사용자에게 텍스트가 점진적으로 나타나는 UX에 적합합니다.

항상 `(Iterator[StreamChunk], meta)` 튜플을 반환합니다. 캐시 히트든 미스든 동일한 방식으로 chunk를 순회합니다.

In [12]:
from llm_request import llm_request_stream

llm_input_stream = {
        "contents": (
            "다음 주제에 대해 3000자 이상의 상세한 보고서를 작성해줘.\n\n"
            "주제: 인공지능(AI)의 발전 역사와 미래 전망\n\n"
            "1. 1950년대~1970년대 AI 태동기 (튜링 테스트, 퍼셉트론, 첫 번째 AI 겨울)\n"
            "2. 1980년대~1990년대 전문가 시스템과 두 번째 AI 겨울\n"
            "3. 2000년대~2010년대 머신러닝 부활, 딥러닝 혁명 (ImageNet, AlphaGo)\n"
            "4. 2020년대 대규모 언어모델(GPT, Gemini 등)과 생성형 AI\n"
            "5. 향후 10년간 AI가 금융, 의료, 교육 분야에 미칠 영향 예측\n\n"
            "각 시대별로 핵심 인물, 주요 기술, 한계점, 사회적 반응을 포함해줘."
        ),
        "model": "gemini-2.5-flash",
        "temperature": 0.7,
        "max_tokens": 4000,
}

stream, meta = llm_request_stream(
    llm_input_stream,
    cache_mode=True,
    save_mode=True,
)
chunk_count = 0
total_text = []
for chunk in stream:
    total_text.append(chunk.text or "")
    chunk_count += 1
    if chunk_count % 5 == 0:
        t = chunk.text or ""
        print(f"  [chunk #{chunk_count:3d}] ...{t[:60]!r}", flush=True)
print(f"\n총 {chunk_count}개 chunk 수신, {''.join(total_text).__len__()}자")
print(f"cache_hit: {meta['cache_hit']}, source: {meta['source']}")


  [chunk #  5] ...'1970년대: AI 태동기 (튜링 테스트, 퍼셉트론, 첫 번째 AI 겨울)\n\n**핵심 인물:**\n*   **'
  [chunk # 10] ..."크 로젠블랫(Frank Rosenblatt):** '퍼셉트론(Perceptron)'을 개발하여 초기 신경망 "
  [chunk # 15] ...'내는 간단한 학습 알고리즘입니다. 이미지 인식 등 특정 문제에서 초기 성공을 거두었습니다.\n*   **ELI'
  [chunk # 20] ...' 신경망 연구에 큰 좌절을 안겨주었습니다.\n*   **지식 표현의 어려움:** 복잡한 현실 세계의 지식을 모'
  [chunk # 25] ...'1990년대: 전문가 시스템과 두 번째 AI 겨울\n\n**핵심 인물:**\n*   **에드워드 파이겐바움(Edw'
  [chunk # 30] ...'1970년대 후반):** 혈액 감염 진단 및 치료법을 제안하는 의료 전문가 시스템.\n    *   **XCO'
  [chunk # 35] ...'화되어 있었기 때문에 다른 분야로의 확장이 어려웠습니다.\n*   **고비용:** 시스템 개발 및 유지보수 비'

총 38개 chunk 수신, 3718자
cache_hit: False, source: live


---
## 5. `llm_request_stream_async` — 스트리밍 LLM 호출 (async)

`llm_request_stream`의 async 버전입니다. `await`로 호출하면 `(AsyncIterator[StreamChunk], meta)` 튜플을 반환하고 `async for`로 chunk를 순회합니다.

In [13]:
from llm_request import llm_request_stream_async

llm_input_async_stream = {
        "contents": (
            "다음 주제에 대해 3000자 이상의 상세한 기술 분석 보고서를 작성해줘.\n\n"
            "주제: 블록체인 기술의 진화와 산업별 적용 현황\n\n"
            "1. 블록체인 1.0 — 비트코인과 분산 원장의 탄생 (합의 알고리즘, 작업증명)\n"
            "2. 블록체인 2.0 — 이더리움과 스마트 컨트랙트 혁명 (DeFi, NFT)\n"
            "3. 블록체인 3.0 — 확장성 솔루션 (Layer 2, 샤딩, 지분증명 전환)\n"
            "4. 금융 분야: CBDC, 토큰증권(STO), 크로스보더 결제 사례\n"
            "5. 물류·공급망: 원산지 추적, IoT 연계, 위변조 방지\n"
            "6. 의료: 전자건강기록(EHR) 공유, 임상시험 데이터 무결성\n"
            "7. 한계점과 향후 과제 (에너지 문제, 규제, 프라이버시)\n\n"
            "각 섹션마다 구체적인 프로젝트/기업 사례를 2개 이상 포함해줘."
        ),
        "model": "gemini-2.5-flash",
        "temperature": 0.7,
        "max_tokens": 4000,
}

stream, meta = await llm_request_stream_async(
    llm_input_async_stream,
    cache_mode=True,
    save_mode=True,
)
chunk_count = 0
total_text = []
async for chunk in stream:
    total_text.append(chunk.text or "")
    chunk_count += 1
    if chunk_count % 5 == 0:
        t = chunk.text or ""
        print(f"  [chunk #{chunk_count:3d}] ...{t[:60]!r}", flush=True)
print(f"\n총 {chunk_count}개 chunk 수신, {''.join(total_text).__len__()}자")
print(f"cache_hit: {meta['cache_hit']}, source: {meta['source']}")


  [chunk #  5] ...' 창출할 수 있는 기반을 제공합니다.\n\n본 보고서는 블록체인 기술의 세대별 진화를 심층적으로 분석하고, 각 '
  [chunk # 10] ...'. 이 원장은 블록(Block)이라는 단위로 묶여 이전 블록과 암호학적으로 연결되며, 한번 기록된 정보는 사'
  [chunk # 15] ...'\n\n*   **보안성:** PoW는 51% 공격(네트워크 해시 파워의 51% 이상을 장악하여 거래를 조작하는'
  [chunk # 20] ...'.\n    *   **기술적 특징:** 고정된 발행량(2,100만 개), 반감기(Halving)를 통한 공급'
  [chunk # 25] ...'56)을 사용하여 일반 CPU/GPU 채굴에 더 친화적으로 설계되었습니다. 총 발행량도 비트코인의 4배인 8'
  [chunk # 30] ...'로 발전시켰습니다.\n\n#### 2.1. 스마트 컨트랙트의 개념과 혁신\n\n스마트 컨트랙트는 특정 조건이 충족되'

총 33개 chunk 수신, 3017자
cache_hit: False, source: live


---
## 7. `done_check` — 캐시 키 생성 + DB 조회

`llm_input`을 정규화 → SHA-256 해시하여 캐시 키를 생성하고, MongoDB에 이미 처리된 결과가 있는지 조회합니다.

`llm_request` 내부에서도 동일한 흐름으로 동작합니다.

In [14]:
from llm_request import done_check

llm_input_done = {
    "contents": "대한민국의 수도는?",
    "model": "gemini-3-flash-preview",
    "temperature": 0,
    "max_tokens": 200,
}

# ① 캐시 키만 생성 (DB 조회 없이)
dc = done_check(llm_input_done, lookup_cache=False)
print(f"cache_key:    {dc.cache_key}")    # SHA-256 (64자)
print(f"canonical:    {dc.canonical_json[:60]}...")  # 정규화된 입력
print(f"collection:   {dc.collection}")
print(f"cached:       {dc.cached}")      # None (조회 안 했으므로)

print()

# ② 캐시 조회까지 수행
dc2 = done_check(llm_input_done, lookup_cache=True)
if dc2.cached:
    print(f"캐시 히트! text: {dc2.cached['result']['text'][:60]}")
else:
    print("캐시 미스 — 저장된 결과 없음")
print(f"cache_error:  {dc2.cache_error}")  # DB 연결 오류 시 True


cache_key:    019b1c02b84bb8e71e957c8a3efa82c1e6d3684a9d25e05796ec09f9a0378003
canonical:    {"cached_content":"","contents":"대한민국의 수도는?","file_checksums...
collection:   agent_llm
cached:       None

캐시 히트! text: 대한민국의 수도는 **서울**입니다.
cache_error:  False


### 캐시 키 동작 원리

`llm_input`에서 15개 필드 전부 추출(미입력/None → `""`) → 값 내부 정규화 → 키 정렬·JSON 직렬화 → SHA-256 해시

**캐시 키에 포함되는 필드:**
`contents`, `system`, `history`, `model`, `max_tokens`, `temperature`, `cached_content`, `service_name`, `seed`, `output`, `schema`, `thinking`, `tools`, `tool_config`, `media_resolution`

이 중 하나라도 다르면 **다른 캐시 키**가 됩니다.


### 7-4. 반환값 구조 — `DoneCheckResult`

```python
@dataclass(frozen=True)
class DoneCheckResult:
    collection: str              # 저장 대상 MongoDB 컬렉션명
    cache_key: str               # SHA-256 해시 (64자)
    canonical_json: str          # 정규화된 입력 JSON 문자열
    cached: dict | None          # 캐시 히트 시 저장된 문서, 미스 시 None
    cache_error: bool            # DB 조회 에러 발생 여부
```

---
## 8. `fetch_by_cache_key` — cache_key로 직접 캐시 결과 조회

이미 `done_check`으로 받은 `cache_key`를 가지고 있을 때, `llm_input` 재구성 없이 **cache_key만으로** 캐시 결과를 조회합니다.

- `as_stream=False` (기본): `(LLMResponse, meta)` 반환 — `llm_request`와 동일한 형식
- `as_stream=True`: `(Iterator[StreamChunk], meta)` 반환 — `llm_request_stream`과 동일한 형식
- 캐시 키에 해당하는 문서가 없으면 `KeyError` 발생

In [15]:
from llm_request import done_check, fetch_by_cache_key

# ① done_check으로 캐시 키 확보
dc = done_check(
    {"contents": "대한민국의 수도는?", "model": "gemini-3-flash-preview", "temperature": 0, "max_tokens": 200},
    lookup_cache=False,
)
cache_key = dc.cache_key
print(f"cache_key: {cache_key}")

# ② cache_key만으로 결과 조회 (일반)
response, meta = fetch_by_cache_key(cache_key)
print(f"text: {response.text[:60]}")
print(f"source: {meta['source']}")  # 'cache'

# ③ 스트리밍 형식으로 조회
stream_iter, meta_s = fetch_by_cache_key(cache_key, as_stream=True)
for chunk in stream_iter:
    print(f"stream chunk: {chunk.text[:60]}")

cache_key: 019b1c02b84bb8e71e957c8a3efa82c1e6d3684a9d25e05796ec09f9a0378003
text: 대한민국의 수도는 **서울**입니다.
source: cache
stream chunk: 대한민국의 수도는 **서울**입니다.


In [16]:
from llm_request import done_check, fetch_by_cache_key

# ① done_check으로 캐시 키 확보
dc = done_check(
    {"contents": "대한민국의 수도는?", "model": "gemini-3-flash-preview", "temperature": 1.1, "max_tokens": 200},
    lookup_cache=False,
)
cache_key = dc.cache_key
print(f"cache_key: {cache_key}")

# ② cache_key만으로 결과 조회 (일반)
response, meta = fetch_by_cache_key(cache_key)
print(f"text: {response.text[:60]}")
print(f"source: {meta['source']}")  # 'cache'

# ③ 스트리밍 형식으로 조회
stream_iter, meta_s = fetch_by_cache_key(cache_key, as_stream=True)
for chunk in stream_iter:
    print(f"stream chunk: {chunk.text[:60]}")

캐시 조회 실패: cache_key=d7bf77af990c8337... 에 해당하는 문서 없음


cache_key: d7bf77af990c8337dc274e622062af7ff1bc48a015516125481be1ee4009236c


KeyError: "cache_key 'd7bf77af990c8337...' 에 해당하는 캐시 문서가 없습니다."

---
# Appendix: 에이전트 개발 패턴

`llm_request`를 사용하는 에이전트 개발 시 자주 쓰는 패턴입니다.

## 패턴 A: 단건 처리 — `llm_request_async`

단건 LLM 호출 + 캐시. 에이전트 내 단일 LLM 호출의 기본 패턴입니다.

In [17]:
from llm_request import llm_request_async

response_a, meta_a = await llm_request_async(
    {
        "contents": "삼성전자 반도체 사업 현황을 두 줄로 요약해줘.",
        "model": "gemini-3-flash-preview",
        "temperature": 0,
        "max_tokens": 150,
        "service_name": "fund_comment",
    },
    cache_mode=True,
    save_mode=True,
)

print(response_a.text)
print(f"source:    {meta_a['source']}")       # 'live' 또는 'cache'
print(f"cache_hit: {meta_a['cache_hit']}")
if response_a.usage:
    u = response_a.usage
    print(f"tokens:    total={u.total_tokens} (in={u.input_tokens}, out={u.output_tokens})")


삼성전자의 반도체
source:    live
cache_hit: False
tokens:    total=164 (in=18, out=6)


---
## 패턴 B: N건 병렬 — `llm_request_async` + 타이밍 로그

스트리밍 대신 **응답 한 덩어리**로 받고, **시간 로그 + 요약 표**로 비동기가 잘 도는지 보기 쉽게 했습니다.

- 펀드마다 **DB 조회 지연**을 다르게 두어 (`0.4s × 인덱스`) LLM 요청 시각이 어긋납니다.
- 프롬프트에 **짧은 단락 요청 + `max_tokens` 여유**를 두어 LLM이 조금 더 걸리게 했습니다.
- 로그에 `[경과초] 코드 DB완료` → `LLM시작` → `LLM끝(소요초)` 가 찍혀, **겹쳐 진행**되는지 눈으로 확인할 수 있습니다.

```
005930: task → DB(즉시) → [LLM 2~4s] ─────────────┐
000660: task → DB(0.4s) → [LLM ...]              ├─ as_completed
035420: task → DB(0.8s) → [LLM ...]              │
068270: task → DB(1.2s) → [LLM ...]              ┘
```

In [18]:
import asyncio
import time

from llm_request import llm_request_async

FUNDS = [
    {"code": "005930", "name": "삼성전자",  "sector": "반도체"},
    {"code": "000660", "name": "SK하이닉스", "sector": "반도체"},
    {"code": "035420", "name": "NAVER",      "sector": "IT"},
    {"code": "068270", "name": "셀트리온",   "sector": "바이오"},
]

# 펀드별 DB 조회 지연(초) — LLM 요청 시각을 어긋나게 해 비동기 로그가 겹쳐 보이게 함
DB_STAGGER_SEC = 0.42


async def process_fund(fund: dict, idx: int, wall0: float) -> dict:
    """DB 조회(시뮬) 후 LLM 호출. wall0 기준 경과 시간 로그."""
    code = fund["code"]

    def rel() -> float:
        return time.perf_counter() - wall0

    print(f"[{rel():5.2f}s] {code} task 시작")

    # 실제 에이전트: fund_data = await db.fetch(fund["code"])
    await asyncio.sleep(DB_STAGGER_SEC * idx)
    t_db = rel()
    print(f"[{t_db:5.2f}s] {code} DB 조회 완료")

    fund_data = f"{fund['name']}({fund['sector']})"

    t_in = rel()
    print(f"[{t_in:5.2f}s] {code} LLM 호출 시작")

    t_llm_wall = time.perf_counter()
    response, meta = await llm_request_async(
        {
            "contents": (
                f"{fund_data} 펀드에 대해 투자 관점에서 "
                f"2~3문장으로 코멘트를 작성해줘. "
                f"리스크와 기회를 각각 한 문장씩 포함해줘."
            ),
            "model": "gemini-3-flash-preview",
            "temperature": 0.4,
            "max_tokens": 1350,
            "service_name": "fund_comment",
        },
        cache_mode=True,
        save_mode=True,
    )
    llm_sec = time.perf_counter() - t_llm_wall
    t_out = rel()
    print(
        f"[{t_out:5.2f}s] {code} LLM 응답 수신 "
        f"(순수 LLM {llm_sec:.2f}s, {len(response.text or '')}자, source={meta['source']})",
        f"         {code} >> {(response.text or '')[:80]}{'…' if len(response.text or '') > 80 else ''}"
    )

    return {
        "code": code,
        "text": response.text,
        "source": meta["source"],
        "cache_key": meta["cache_key"],
        "t_db_done": t_db,
        "t_llm_start": t_in,
        "t_llm_end": t_out,
        "llm_sec": llm_sec,
    }


print("=" * 72)
print(f"{len(FUNDS)}개 펀드 병렬 (DB 지연 {DB_STAGGER_SEC}s×인덱스, max_tokens=1350)")
print("타임스탬프는 아래 '패턴 B 시작' 시각 기준 경과 초입니다.")
print("=" * 72)

wall0 = time.perf_counter()
tasks = [asyncio.create_task(process_fund(f, i, wall0)) for i, f in enumerate(FUNDS)]
many_results = []

for coro in asyncio.as_completed(tasks):
    result = await coro
    wall = time.perf_counter() - wall0
    many_results.append(result)
    print(f"  >> as_completed [{wall:5.2f}s] {result['code']} 전체 Task 종료")

print("=" * 72)
print(f"벽시계 전체: {time.perf_counter() - wall0:.2f}s")



4개 펀드 병렬 (DB 지연 0.42s×인덱스, max_tokens=1350)
타임스탬프는 아래 '패턴 B 시작' 시각 기준 경과 초입니다.
[ 0.00s] 005930 task 시작
[ 0.00s] 000660 task 시작
[ 0.00s] 035420 task 시작
[ 0.00s] 068270 task 시작
[ 0.00s] 005930 DB 조회 완료
[ 0.00s] 005930 LLM 호출 시작
[ 0.42s] 000660 DB 조회 완료
[ 0.42s] 000660 LLM 호출 시작
[ 0.84s] 035420 DB 조회 완료
[ 0.84s] 035420 LLM 호출 시작
[ 1.26s] 068270 DB 조회 완료
[ 1.26s] 068270 LLM 호출 시작
[ 4.47s] 005930 LLM 응답 수신 (순수 LLM 4.47s, 163자, source=live)          005930 >> 삼성전자는 AI 산업 확장에 따른 HBM(고대역폭메모리) 수요 급증과 반도체 업황 회복에 힘입어 강력한 실적 반등의 기회를 맞이하고 있습니다. …
  >> as_completed [ 4.47s] 005930 전체 Task 종료
[ 5.56s] 035420 LLM 응답 수신 (순수 LLM 4.72s, 264자, source=live)          035420 >> NAVER 펀드는 생성형 AI '하이퍼클로바X'를 활용한 서비스 고도화와 웹툰 등 글로벌 콘텐츠 사업의 성장이 중장기적인 기업 가치 상승을 이끌…
  >> as_completed [ 5.56s] 035420 전체 Task 종료
[ 6.12s] 068270 LLM 응답 수신 (순수 LLM 4.86s, 173자, source=live)          068270 >> 셀트리온 펀드는 통합 셀트리온 출범에 따른 원가 구조 개선과 신약 '짐펜트라'의 미국 시장 안착을 통한 실적 퀀텀 점프가 기대되는 매력적인 투자…
  >> as_completed [ 6.12s] 068270 전체 Task 종료
[ 7

### 결과 확인 — 요약 표

같은 **패턴 B 시작 시각** 기준으로 DB 완료·LLM 구간·순수 LLM 소요시간을 한눈에 봅니다.  
`LLM시작` 시각이 DB끝 직후에 연달아 찍히면 비동기 병렬 진행이 확인됩니다. 동시 호출 제한은 llm-handler(`gemini_max_concurrency`)가 담당합니다.
결과를 통해 앞선 과정(예시 DB)이 종료되는 즉시 llm_request_async를 실행하고, 응답을 바로 받을 수 있습니다.

In [19]:
rows = sorted(many_results, key=lambda x: x["code"])
hdr = f"{'코드':<8} {'DB끝(s)':>8} {'LLM시작(s)':>10} {'LLM끝(s)':>9} {'순수LLM(s)':>10} {'source':<8}"
print(hdr)
print("-" * len(hdr))
for r in rows:
    print(
        f"{r['code']:<8} "
        f"{r['t_db_done']:8.2f} "
        f"{r['t_llm_start']:10.2f} "
        f"{r['t_llm_end']:9.2f} "
        f"{r['llm_sec']:10.2f} "
        f"{r['source']:<8}"
    )

print()
for r in rows:
    preview = r["text"].replace("\n", " ")[:100]
    print(f"[{r['code']}] {preview}{'…' if len(r['text']) > 100 else ''}")

코드         DB끝(s)   LLM시작(s)   LLM끝(s)   순수LLM(s) source  
----------------------------------------------------------
000660       0.42       0.42      7.21       6.79 live    
005930       0.00       0.00      4.47       4.47 live    
035420       0.84       0.84      5.56       4.72 live    
068270       1.26       1.26      6.12       4.86 live    

[000660] SK하이닉스는 AI 반도체의 핵심인 HBM(고대역폭 메모리) 시장에서의 독보적인 기술 우위를 바탕으로 강력한 실적 성장 모멘텀을 보유하고 있습니다. 다만, 메모리 업황 특유의 높은…
[005930] 삼성전자는 AI 산업 확장에 따른 HBM(고대역폭메모리) 수요 급증과 반도체 업황 회복에 힘입어 강력한 실적 반등의 기회를 맞이하고 있습니다. 다만, 글로벌 파운드리 경쟁 심화와 …
[035420] NAVER 펀드는 생성형 AI '하이퍼클로바X'를 활용한 서비스 고도화와 웹툰 등 글로벌 콘텐츠 사업의 성장이 중장기적인 기업 가치 상승을 이끌 핵심 기회 요인입니다. 다만, 유튜…
[068270] 셀트리온 펀드는 통합 셀트리온 출범에 따른 원가 구조 개선과 신약 '짐펜트라'의 미국 시장 안착을 통한 실적 퀀텀 점프가 기대되는 매력적인 투자처입니다. 다만, 글로벌 바이오시밀러…


---
## 정리

테스트 데이터를 삭제합니다.

In [21]:
from llm_request import done_check

mdb = MongoDBHandler()
col = mdb.get_database("kw_ai_dev", "agent_llm")
deleted = 0

# 섹션 1 기본 예제
for llm_in in [llm_input, llm_input_no_cache, llm_input_system]:
    dc = done_check(llm_in, lookup_cache=False)
    res = col.delete_one({"_id": dc.cache_key})
    deleted += res

# 섹션 2 async 예제
if meta_async.get("cache_key"):
    res = col.delete_one({"_id": meta_async["cache_key"]})
    deleted += res

# 섹션 3 PDF (Part.from_file + Files API)
if meta.get("cache_key"):
    res = col.delete_one({"_id": meta["cache_key"]})
    deleted += res
if meta2.get("cache_key"):
    res = col.delete_one({"_id": meta2["cache_key"]})
    deleted += res

# 섹션 4, 5 stream 예제
for llm_in in [llm_input_stream, llm_input_async_stream]:
    dc = done_check(llm_in, lookup_cache=False)
    res = col.delete_one({"_id": dc.cache_key})
    deleted += res

# Appendix 패턴 A
if meta_a.get("cache_key"):
    res = col.delete_one({"_id": meta_a["cache_key"]})
    deleted += res

# Appendix 패턴 B (cache_key는 process_fund 결과에서 직접 참조)
for r in many_results:
    if r.get("cache_key"):
        res = col.delete_one({"_id": r["cache_key"]})
        deleted += res

print(f"정리 완료: {deleted}건 삭제")


정리 완료: 8건 삭제
